In [ ]:
# Copyright (c) 2020-2022, NVIDIA CORPORATION.  All rights reserved.
#
# NVIDIA CORPORATION and its licensors retain all intellectual property
# and proprietary rights in and to this software, related documentation
# and any modifications thereto.  Any use, reproduction, disclosure or
# distribution of this software and related documentation without an express
# license agreement from NVIDIA CORPORATION is strictly prohibited.

In [ ]:
import sys
import numpy as np
from omni.isaac.kit import SimulationApp

simulation_app = SimulationApp(launch_config={"headless": True})
import omni

simulation_app.update()
omni.usd.get_context().new_stage()
simulation_app.update()


In [ ]:
from omni.isaac.synthetic_utils import SyntheticDataHelper
from omni.isaac.core.objects import VisualCuboid, DynamicCuboid

viewport = omni.kit.viewport_legacy.get_default_viewport_window()
sd_helper = SyntheticDataHelper()
sensor_names = [
    "rgb",
    "depth",
    "boundingBox2DTight",
    "boundingBox2DLoose",
    "instanceSegmentation",
    "semanticSegmentation",
    "boundingBox3D",
    "camera",
    "pose",
]

VisualCuboid(
    prim_path="/new_cube_1",
    name="visual_cube",
    position=np.array([0, 0, 0.5]),
    scale=np.array([1, 1, 1]),
    color=np.array([255, 255, 255]),
)


In [ ]:
simulation_app.update()
sd_helper.initialize(sensor_names=sensor_names, viewport=viewport)
gt = sd_helper.get_groundtruth(sensor_names=sensor_names, viewport=viewport, verify_sensor_init=False)
print(gt)
print(gt["rgb"].size)
if gt["rgb"].size != 1280 * 720 * 4:
    raise ValueError(f"RGB buffer has size of {gt['rgb'].size} which is not {1280*720*4}")


In [ ]:
import omni.graph.core as og
import omni

from omni.syntheticdata import sensors

simulation_app.update()
viewport = omni.kit.viewport_legacy.get_default_viewport_window()
import omni.syntheticdata._syntheticdata as sd

sensors.enable_sensors(viewport, [sd.SensorType.DistanceToImagePlane])
simulation_app.update()
graph = og.ObjectLookup.graph("/Render/PostProcess/SDGPipeline")
raw_node = og.ObjectLookup.node(
    "/Render/PostProcess/SDGPipeline/RenderProduct_Viewport_DistanceToImagePlaneSDExportRawArray"
)
swh_attr = og.Controller.attribute("outputs:swhFrameNumber", raw_node)
first = og.DataView.get(swh_attr)
simulation_app.update()
second = og.DataView.get(swh_attr)

if first + 1 != second:
    raise ValueError(f"swh frame numbers {first}, {second} should be one apart")


In [ ]:
# Cleanup application
simulation_app.close()
